In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

## Libraries and data

In [ ]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.plot import set_dark_theme
from skforecast.preprocessing import (
    reshape_series_long_to_dict, 
    reshape_exog_long_to_dict, 
    RollingFeatures
)
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_foundation,
    bayesian_search_foundation
)

# Load time series of multiple lengths and exogenous variables
# ==============================================================================
series = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series.csv'
)
exog = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series_exog.csv'
)

series['timestamp'] = pd.to_datetime(series['timestamp'])
exog['timestamp'] = pd.to_datetime(exog['timestamp'])

display(series.head(3))
print("")
display(exog.head(3))


# Transform series and exog to dictionaries
# ==============================================================================
series_dict = reshape_series_long_to_dict(
    data      = series,
    series_id = 'series_id',
    index     = 'timestamp',
    values    = 'value',
    freq      = 'D'
)

exog_dict = reshape_exog_long_to_dict(
    data      = exog,
    series_id = 'series_id',
    index     = 'timestamp',
    freq      = 'D'
)


# Drop some exogenous variables for series 'id_1000' and 'id_1003'
# ==============================================================================
# Some exogenous variables are intentionally omitted for series 1 and 3 to illustrate that each series can use a different set of exogenous variables.
exog_dict['id_1000'] = exog_dict['id_1000'].drop(columns=['air_temperature', 'wind_speed'])
exog_dict['id_1003'] = exog_dict['id_1003'].drop(columns=['cos_day_of_week'])


# Partition data in train and test
# ==============================================================================
end_train = '2016-07-31 23:59:00'
series_dict_train = {k: v.loc[: end_train,] for k, v in series_dict.items()}
exog_dict_train   = {k: v.loc[: end_train,] for k, v in exog_dict.items()}
series_dict_test  = {k: v.loc[end_train:,] for k, v in series_dict.items()}
exog_dict_test    = {k: v.loc[end_train:,] for k, v in exog_dict.items()}


# Description of each partition
# ==============================================================================
for k in series_dict.keys():
    print(f"{k}:")
    try:
        print(
            f"\tTrain: len={len(series_dict_train[k])}, {series_dict_train[k].index[0]}"
            f" --- {series_dict_train[k].index[-1]}"
        )
    except IndexError:
        print("\tTrain: len=0")
    try:
        print(
            f"\tTest : len={len(series_dict_test[k])}, {series_dict_test[k].index[0]}"
            f" --- {series_dict_test[k].index[-1]}"
        )
    except IndexError:
        print("\tTest : len=0")


# Exogenous variables for each series
# ==============================================================================
for k in series_dict.keys():
    print(f"{k}:")
    try:
        print(f"\t{exog_dict[k].columns.to_list()}")
    except IndexError:
        print("\tNo exogenous variables")


# Fit forecaster
# ==============================================================================
model_id = "google/timesfm-3.0-pytorch"
model_id="autogluon/chronos-2-small"

estimator = FoundationModel(model_id=model_id, context_length=500)
forecaster = ForecasterFoundation(estimator=estimator)
forecaster.fit(series=series_dict_train, exog=exog_dict_train)

# Predict
# ==============================================================================
predictions = forecaster.predict(steps=5, exog=exog_dict_test)
predictions.head(9)


# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 24,
         initial_train_size = "2016-07-31 23:59:00",
     )

metrics_levels, backtest_predictions = backtesting_foundation(
    forecaster            = forecaster,
    series                = series_dict,
    exog                  = exog_dict,
    cv                    = cv,
    levels                = None,
    metric                = "mean_absolute_error",
    add_aggregated_metric = True,
    suppress_warnings     = True
)

display(metrics_levels)
backtest_predictions

,series_id,timestamp,value
0,id_1000,2016-01-01,1012.500694
1,id_1000,2016-01-02,1158.500099
2,id_1000,2016-01-03,983.000099


,series_id,timestamp,sin_day_of_week,cos_day_of_week,air_temperature,wind_speed
0,id_1000,2016-01-01,-0.433884,-0.900969,6.416639,4.040115
1,id_1000,2016-01-02,-0.974928,-0.222521,6.366474,4.530395
2,id_1000,2016-01-03,-0.781831,0.623490,6.555272,3.273064


╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Series 'id_1003' is incomplete. NaNs have been introduced after setting the          │
│ frequency.                                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ c:\Users\Joaquin\miniconda3\envs\skforecast_24_py13\Lib\site-packages\skforecast\pre │
│ processing\_preprocessing.py:531                                                     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

id_1000:
	Train: len=213, 2016-01-01 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1001:
	Train: len=30, 2016-07-02 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1002:
	Train: len=183, 2016-01-01 00:00:00 --- 2016-07-01 00:00:00
	Test : len=0
id_1003:
	Train: len=213, 2016-01-01 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1004:
	Train: len=91, 2016-05-02 00:00:00 --- 2016-07-31 00:00:00
	Test : len=31, 2016-08-01 00:00:00 --- 2016-08-31 00:00:00
id_1000:
	['sin_day_of_week', 'cos_day_of_week']
id_1001:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']
id_1002:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']
id_1003:
	['sin_day_of_week', 'air_temperature', 'wind_speed']
id_1004:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']


╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series ['id_1002'] has been reindexed to match the expected forecast      │
│ horizon. Missing timestamps were filled with NaN.                                    │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location : C:\Users\Joaquin\AppData\Local\Temp\ipykernel_32804\2787424888.py:111     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── LicenseWarning ───────────────────────────────────╮
│ The weights for 'google/timesfm-3.0-pytorch' are released under TimesFM              │
│ Non-Commercial License v1.0, which restricts their use to non-commercial or          │
│ non-production purposes. Review the license before deploying. See                    │
│ https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE.                 │
│                                                                                      │
│ Category : skforecast.exceptions.LicenseWarning                                      │
│ Location :                                                                           │
│ c:\Users\Joaquin\miniconda3\envs\skforecast_24_py13\Lib\site-packages\skforecast\fou │
│ ndation\_adapters.py:1283                                                            │
│ Suppress : warnings.simplefilter('ignore', category=LicenseWarning)                  │
╰──────────────────────────────────────────────────────────────────────────────────────╯

ValueError: all input arrays must have the same shape